# المُنسِّق — مساعد مكتب جمعية بمشرف وعمّال

**الاسم الكامل:** Abdulaziz Mulia (عبدالعزيز مُليا)

**البرنامج:** SDAIA — بناء أنظمة وكلاء الذكاء الاصطناعي، 16–20 أغسطس 2026

**المسار:** A — مشرف (Supervisor) وثلاثة عمّال: التقويم، والمعرفة، والمراسلات.

> **تنويه**: كل ما في `data/corpus/` **وثائق تركيبية** أُلّفت لهذا المشروع
> التدريبي، ولا تمثّل سياسة فعلية لأي جهة. ولا يقع في المشروع إرسال بريد
> حقيقي — «الإرسال» كتابةٌ في صندوق صادر محلي تحت `data/outbox/`.

**كيف يُقرأ هذا النوتبوك**: المنطق كله يسكن `src/munassiq/` فيبقى قابلًا
للاختبار بـ`pytest`، وهذه الخلايا **دليل تشغيل** يستورد تلك الواجهات ويعرض
مخرجاتها بترتيب التنفيذ — خليةٌ أو أكثر لكل قسم من أقسام الروبرك الثمانية،
وخريطةٌ تربطها كلها في الخاتمة.


## الإعداد — البيئة والتتبع وذاكرة العرض

`src/` يُضاف إلى مسار الاستيراد، و`config.py` يحمّل `.env` **نسبيًا** من جذر
المشروع: لا مسار مطلق في أي ملف متتبَّع (المسار المطلق يكشف اسم المستخدم
وبنية الجهاز)، ولا تُطبع قيمة أي مفتاح هنا ولا في أي خلية بعدها. ثم
`assert_tracing_configured()` يفشل مبكرًا إن كان التتبع مطفأً — بدل أن ينتهي
النوتبوك كله ثم يُكتشف أن مشروع LangSmith فارغ.

وذاكرة العرض تُصفَّر إلى ملفٍ جديد في كل تشغيلة: بلا تصفير تتراكم ذكريات
التشغيلات السابقة، فيصير «التذكّر» في القسم 4 أثرَ ماضٍ لا دليلَ هذه التشغيلة.


In [1]:
import shutil
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "src" / "munassiq").is_dir(), (
    "شغّل النوتبوك من جذر المشروع — المجلد الذي فيه src/ و data/"
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# تحذيرات المكتبات تُطبع مع **مسار الملف المطلق** داخل ‎.venv، وهو يكشف اسم
# المستخدم وبنية الجهاز ويبقى محفوظًا في مخرجات النوتبوك. الكتم قرار أمن لا
# تجميل — وبوابة tools/leak_scan.py تحرس ما بقي.
warnings.filterwarnings("ignore")

import os  # noqa: E402
from munassiq.config import DEFAULT_MODELS, DEFAULT_PROVIDER  # noqa: E402  بعد ضبط sys.path
from munassiq.memory import reset_memory_for_tests  # noqa: E402
from munassiq.tracing import assert_tracing_configured  # noqa: E402

# ذاكرة عرضٍ جديدة في كل تشغيلة (ملفات ‎*.sqlite مُتجاهَلة في git).
DEMO_STATE_DIR = PROJECT_ROOT / "data" / "notebook-state"
shutil.rmtree(DEMO_STATE_DIR, ignore_errors=True)
DEMO_STATE_DIR.mkdir(parents=True, exist_ok=True)
checkpointer, store = reset_memory_for_tests(DEMO_STATE_DIR)

# التصفير **قبل** بناء الـapp: الـapp يلتقط الـcheckpointer والـstore لحظة بنائه.
from munassiq.app import build_app  # noqa: E402

app = build_app()

summary = assert_tracing_configured()
PROVIDER = os.environ.get("MUNASSIQ_PROVIDER", DEFAULT_PROVIDER)
MODEL = os.environ.get("MUNASSIQ_MODEL") or DEFAULT_MODELS[PROVIDER]
print("المزود:", PROVIDER, "| النموذج:", MODEL)
print("التتبع:", summary["flag"], "= true | المشروع:", summary["project"])
print("مفتاح LangSmith موجود:", summary["api_key_present"], "— القيمة لا تُطبع")
print("الذاكرة:", type(checkpointer).__name__, "+", type(store).__name__)


المزود: openrouter | النموذج: openai/gpt-oss-120b
التتبع: LANGCHAIN_TRACING_V2 = true | المشروع: munassiq-capstone
مفتاح LangSmith موجود: True — القيمة لا تُطبع
الذاكرة: SqliteSaver + SqliteStore


## القسم 1 — أساسيات الوكيل: أدوات تستعمل معاملاتها، ومخرجٌ مهيكل

ثلاث أدوات معرَّفة بـ`@tool` في `munassiq/tools.py`، وكل واحدة **تستعمل
معاملاتها فعلًا**: `create_event(title, day)` تُلحق بقائمة `CALENDAR` عنصرًا
مشتقًّا من معاملَيها، فمعاملان مختلفان يعطيان حالةً مختلفة وناتجًا مختلفًا —
وهذا هو الفرق بين نداء أداةٍ حقيقي ودالةٍ تتجاهل ما يُمرَّر إليها.

والمخرج المهيكل نموذج Pydantic هو `TriageDecision` عبر
`with_structured_output`: يعود كائنٌ بحقول مُتحقَّق منها (`worker` من قائمة
مغلقة، `needs_human_approval` منطقي، `summary` نصّي) — لا نصٌّ حر يُفتَّش فيه
بـ`in` ليُستخرج منه قرار.


In [2]:
from munassiq.tools import CALENDAR, create_event, list_events

CALENDAR.clear()  # التقويم حالة في الذاكرة — يُصفَّر ليكون العرض متكررًا

print(create_event.invoke({"title": "اجتماع لجنة المحتوى", "day": "الثلاثاء"}))
print(create_event.invoke({"title": "ورشة المتطوعين", "day": "الخميس"}))
print("أثر المعاملات في الحالة:", CALENDAR)
print(list_events.invoke({}))


أُضيف الموعد «اجتماع لجنة المحتوى» يوم الثلاثاء إلى التقويم.
أُضيف الموعد «ورشة المتطوعين» يوم الخميس إلى التقويم.
أثر المعاملات في الحالة: [{'title': 'اجتماع لجنة المحتوى', 'day': 'الثلاثاء'}, {'title': 'ورشة المتطوعين', 'day': 'الخميس'}]
مواعيد التقويم:
- «اجتماع لجنة المحتوى» يوم الثلاثاء
- «ورشة المتطوعين» يوم الخميس


In [3]:
import json

from munassiq.tools import TriageDecision, triage

decision = triage("احجز اجتماعًا يوم الأحد")

print("نوع المخرج:", type(decision).__name__, "— نموذج Pydantic لا نص")
print("الحقول المُتحقَّق منها:", list(TriageDecision.model_fields))
print(json.dumps(decision.model_dump(), ensure_ascii=False, indent=2))


نوع المخرج: TriageDecision — نموذج Pydantic لا نص
الحقول المُتحقَّق منها: ['worker', 'needs_human_approval', 'summary']
{
  "worker": "calendar",
  "needs_human_approval": false,
  "summary": "طلب حجز اجتماع يوم الأحد."
}


## القسم 2 — المشرف والتوجيه: نمط Orchestrator-Worker

البنية هنا **Orchestrator-Worker**: منسِّقٌ واحد (المشرف) يستقبل الطلب
ويفوّضه إلى العامل المختص به، وكل عامل وكيل ReAct كامل لا تُتاح له إلا أدوات
اختصاصه — عامل التقويم لا يرى أداة البريد أصلًا، فلا يقدر أن ينحرف إليها.
حصرٌ بالبنية لا برجاءٍ في نص التعليمات.

والتوجيه **قرار نموذج** لا سلسلة شروط: `create_supervisor` يشتق من اسم كل
عامل أداةَ تسليم `transfer_to_<name>`، فيظهر التفويض في الرسائل نداءَ أداةٍ
صريحًا — وهو الدليل المطبوع أدناه. وprompt المشرف يمنعه نصًّا من الإجابة
بنفسه؛ بغير هذا المنع يميل النموذج إلى تلبية الطلب من عنده، فيخرج جوابٌ معقول
بلا أي نداء أداة والتقويم فارغ.


In [4]:
from munassiq.supervisor import build_supervisor

supervisor = build_supervisor()
routed = supervisor.invoke(
    {"messages": [{"role": "user", "content": "احجز اجتماع لجنة المحتوى يوم الثلاثاء"}]}
)

tool_calls = [
    call["name"]
    for message in routed["messages"]
    for call in (getattr(message, "tool_calls", None) or [])
]
print("كل نداءات الأدوات في الرحلة:", tool_calls)
print("نداءات التسليم:", [name for name in tool_calls if name.startswith("transfer_to_")])
print("التقويم بعد التوجيه:", CALENDAR)
print("رد المشرف:", routed["messages"][-1].content)


كل نداءات الأدوات في الرحلة: ['transfer_to_calendar_agent', 'transfer_back_to_supervisor', 'transfer_to_calendar_agent', 'transfer_back_to_supervisor', 'transfer_to_calendar_agent', 'transfer_back_to_supervisor']
نداءات التسليم: ['transfer_to_calendar_agent', 'transfer_to_calendar_agent', 'transfer_to_calendar_agent']
التقويم بعد التوجيه: [{'title': 'اجتماع لجنة المحتوى', 'day': 'الثلاثاء'}, {'title': 'ورشة المتطوعين', 'day': 'الخميس'}, {'title': 'اجتماع لجنة المحتوى', 'day': 'الثلاثاء'}, {'title': 'اجتماع لجنة المحتوى', 'day': 'الثلاثاء'}, {'title': 'اجتماع لجنة المحتوى', 'day': 'الثلاثاء'}]
رد المشرف: 


## القسم 3 — RAG: الاختيار بين 2-Step وAgentic وHybrid

ثلاثة أنماط كانت مطروحة: **2-Step RAG** يسترجع مرة واحدة قبل كل جواب ثم
يولّد منه؛ و**Agentic RAG** يجعل الاسترجاع أداةً بيد وكيلٍ يقرر بنفسه متى
يستعلم وبأي صياغة وكم مرة؛ و**Hybrid RAG** وسطٌ بينهما يسترجع دائمًا مرةً
أولى ويترك للوكيل استعلامًا إضافيًا عند الحاجة.

**المختار هنا Agentic RAG**: طلبات مكتب الجمعية مختلطة الطبيعة — حجز موعد،
صياغة رسالة، سؤال عن سياسة — والمشرف يفوّض كل طلب إلى عامله، فلا يصل
الاسترجاعَ إلا ما يستحقه. وأسئلة السياسات نفسها متفاوتة: بعضها تكفيه نتيجة
واحدة، وبعضها يقع جوابه بين وثيقتين فيحتاج استعلامًا ثانيًا بصياغة أخرى، وهذا
قرارٌ لا يصلح تثبيته سلفًا في خطٍّ ثابت.

**والمقابل مذكور بلا تجميل**: 2-Step أبسط وأرخص وأثبت زمنًا — خطوة استرجاع
واحدة معلومة التكلفة — لكنه يسترجع لكل طلب حتى ما لا يحتاج استرجاعًا، فيدفع
كلفة تضمينٍ وسياقٍ في طلب حجز موعد لا علاقة له بالوثائق. وHybrid يخفف ذلك
جزئيًا لكنه يبقي الاسترجاع الأول إجباريًا. وثمن Agentic أن عدد النداءات غير
محدود سلفًا: تأخّرٌ أعلى وتكلفة أقل تنبؤًا، وسلوكٌ يعتمد على حسن التعليمات —
ولذلك حُصر عامل المعرفة بأداةٍ واحدة، وأُلزم نصًّا بالإجابة من المقاطع وحدها
وبقول «لا أجد هذا في وثائق الجمعية» عند غيابها.

**والتضمين متعدد اللغات إلزامًا**: الموديل الافتراضي في fastembed إنجليزي
(`bge-small-en`) وفشل مع العربية، فاسترجع مقاطع لا صلة لها بالسؤال؛ والمعتمد
`paraphrase-multilingual-MiniLM-L12-v2`. والإحماء منفصل عن الاسترجاع لأن أول
نداء تضمين قد ينزّل الموديل — وهذا زمن تحميل لا زمن استرجاع.


In [5]:
from munassiq.rag import build_retriever, warm_up_embeddings

warm_up_embeddings()  # تحميل الموديل معزولًا عمّا يُقاس
retriever = build_retriever()

QUESTION = "كم مدة مراجعة المحتوى قبل النشر؟"
passages = retriever.invoke(QUESTION)

print("عدد المقاطع المسترجَعة:", len(passages))
print("مصدر المقطع الأول:", passages[0].metadata["source"])
print("الحقيقة «ثلاثة أيام عمل» مسترجَعة:",
      any("ثلاثة أيام عمل" in p.page_content for p in passages))
print("---- المقطع الأول ----")
print(passages[0].page_content[:320])


عدد المقاطع المسترجَعة: 3
مصدر المقطع الأول: سياسة-النشر.md
الحقيقة «ثلاثة أيام عمل» مسترجَعة: True
---- المقطع الأول ----
# سياسة نشر المحتوى — جمعية المحتوى الإسلامي (وثيقة تركيبية للتدريب)

> هذه وثيقة تركيبية أُلّفت لأغراض مشروع تدريبي. لا تمثّل سياسة فعلية لأي جهة.

## دورة المراجعة

كل مادة محتوى تمر بمراجعة علمية قبل النشر. مدة مراجعة المحتوى قبل النشر
ثلاثة أيام عمل من تاريخ إحالة المسودة إلى اللجنة العلمية، وتُمدَّد إلى خمسة
أيام 


In [6]:
from munassiq.workers import build_knowledge_agent

knowledge_agent = build_knowledge_agent()
answered = knowledge_agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})

searches = [
    call["name"]
    for message in answered["messages"]
    for call in (getattr(message, "tool_calls", None) or [])
]
print("أدوات ناداها عامل المعرفة (هو من قرر متى):", searches)
print("الجواب:", answered["messages"][-1].content)


أدوات ناداها عامل المعرفة (هو من قرر متى): ['search_policies']
الجواب: مدة مراجعة المحتوى قبل النشر هي **ثلاثة أيام عمل** من تاريخ إحالة المسودة إلى اللجنة العلمية، وتُمدَّد إلى **خمسة أيام عمل** إذا احتوت المادة على استشهادات تحتاج تدقيق مصادر. (المصدر: سياسة-النشر.md)


## القسم 4 — الذاكرة: قصيرة المدى وطويلة المدى، لا نوعٌ واحد بمقياسين

* **قصيرة المدى** — `SqliteSaver` مفتاحه `thread_id`: حالةُ محادثةٍ واحدة،
  تُستأنف بعد إعادة التشغيل لأنها على القرص لا في الذاكرة.
* **طويلة المدى** — Store بفضاء أسماء `("memories", user_id)`: لا يعرف
  الـ`thread` أصلًا، فما يُكتب فيه في محادثة يُقرأ في محادثةٍ أخرى.

وهذا هو الفرق الذي يسقط فيه أكثر المتقدمين: رسائل متراكمة في thread واحد
**ليست** ذاكرة طويلة المدى، بل سياق محادثة. فالدليل أدناه ثلاثي: كتابةٌ في
`nb-thread-1`، ثم `store.search` يُظهر الحقيقة مكتوبةً **خارج** الـthread، ثم
استدعاءٌ من `nb-thread-2` لا يشترك مع الأول في أي رسالة. وحقن الذكريات يقع في
كل رحلة بلا شرط — لو تُرك لأداةٍ يقرر النموذج نداءها لصار التذكّر احتمالًا لا
ضمانًا.


In [7]:
from munassiq.memory import MEMORY_NAMESPACE

USER_ID = "member-001"
THREAD_1 = {"configurable": {"thread_id": "nb-thread-1"}}

written = app.invoke(
    {"request": "تذكّر أن اليوم المفضل لاجتماعاتنا هو الخميس", "user_id": USER_ID},
    THREAD_1,
)
print("رد thread-1:", written["reply"])

# الدليل الصلب: الحقيقة في الـStore نفسه، لا في نص جواب النموذج.
remembered = store.search((MEMORY_NAMESPACE, USER_ID))
print("ما في الـStore بعد thread-1:", [item.value for item in remembered])


رد thread-1: شكرًا لتذكيرك؛ سأضع ذلك في الاعتبار.
ما في الـStore بعد thread-1: [{'fact': 'تذكّر أن اليوم المفضل لاجتماعاتنا هو الخميس'}]


In [8]:
THREAD_2 = {"configurable": {"thread_id": "nb-thread-2"}}

recalled = app.invoke(
    {"request": "ما اليوم المفضل لاجتماعاتنا؟", "user_id": USER_ID}, THREAD_2
)
print("الذكريات المحقونة في thread-2:", recalled["memories_used"])
print("رد thread-2:", recalled["reply"])

# قِصر المدى: دورٌ ثانٍ على thread-2 نفسه يجد أثر الدور الأول في الـcheckpointer.
follow_up = app.invoke(
    {"request": "وما المواعيد المسجّلة في التقويم؟", "user_id": USER_ID}, THREAD_2
)
print("رقم الدور على thread-2:", follow_up["turn"])
print("لقطة الحالة المحفوظة:", app.get_state(THREAD_2).values.get("turn"))


الذكريات المحقونة في thread-2: ['تذكّر أن اليوم المفضل لاجتماعاتنا هو الخميس']
رد thread-2: اليوم المفضَّل لاجتماعاتنا هو **الخميس**.


رقم الدور على thread-2: 2
لقطة الحالة المحفوظة: 2


In [9]:
import inspect

from langgraph.checkpoint.sqlite import SqliteSaver

# ‏from_conn_string **مدير سياق**: توقيعه يعيد Iterator، والخروج من ‎with يغلق
# الاتصال. استعماله خارج ‎with يعطي كائنًا فوق اتصال مغلق — يعمل في السطر الأول
# ويفشل في الثاني. ولذلك يبني memory.py الاتصال بنفسه ويمرّره للباني مباشرة.
print("النوع:", type(SqliteSaver.from_conn_string).__name__)
print("التوقيع:", inspect.signature(SqliteSaver.from_conn_string))

# النمط عبر العمليات (درس Production) — مكتوبٌ ولا يُنفَّذ هنا، فالعرض عمليةٌ
# واحدة: عمليةٌ ثانية تفتح الملف نفسه، تعيد تعريف الـentrypoint نفسه، وتستأنف
# رحلةً موقوفة بـthread_id نفسه:
#
#     with SqliteSaver.from_conn_string("data/munassiq-state.sqlite") as saver:
#         resumed_app = build_app(checkpointer=saver, store=store)
#         resumed_app.invoke(Command(resume="نص معتمد"), {"configurable": {"thread_id": "nb-mail"}})


النوع: method
التوقيع: (conn_string: 'str') -> 'Iterator[SqliteSaver]'


## القسم 5 — الوقوف البشري: interrupt ثم resume

الفعل غير القابل للعكس (إرسالُ بريدٍ باسم الجمعية) لا يقع إلا بعد `interrupt`
يعرض على البشري مسودةً **مصوغةً سلفًا** — فهو يراجع نصًّا لا فراغًا. وموضع
الوقوف جسمُ الـ`@entrypoint` نفسه لا داخل `@task`: الـ`@task` وحدةٌ تُعاد أو
تُستعاد كاملة، فوقوفٌ في وسطها يعني إعادة تنفيذ ما سبقه عند الاستئناف.

والشقّان في **خليتين منفصلتين** عمدًا: الأولى تُثبت أن التنفيذ وقف فعلًا وأن
شيئًا لم يُكتب بعد، والثانية تستأنف بـ`Command(resume=...)`. وما يعود من
الاستئناف يمضي **حرفيًا** إلى صندوق الصادر بلا أي مرور على نموذج — وإلا لم
يعد ما خرج تعديلَ البشري.


In [10]:
MAIL_REQUEST = "أرسل بريدًا للمتطوعين عن تأجيل فعالية السبت"
THREAD_MAIL = {"configurable": {"thread_id": "nb-mail"}}

paused = app.invoke({"request": MAIL_REQUEST, "user_id": USER_ID}, THREAD_MAIL)

print("مفاتيح ناتج الرحلة الموقوفة:", sorted(paused))
payload = paused["__interrupt__"][0].value
print("الفعل المطلوب من البشري:", payload["action"])
print("ملخّص الطلب:", payload["summary"])
print("---- المسودة المعروضة للمراجعة ----")
print(payload["draft"])


مفاتيح ناتج الرحلة الموقوفة: ['__interrupt__']
الفعل المطلوب من البشري: راجع المسودة واعتمدها أو عدّلها
ملخّص الطلب: طلب صياغة بريد إلكتروني للمتطوعين لإبلاغهم بتأجيل فعالية يوم السبت.
---- المسودة المعروضة للمراجعة ----
تحية طيبة،  

نود إبلاغكم بأنه تم تأجيل فعالية السبت المقررة إلى تاريخ لاحق، وسيتم إعلامكم بالموعد الجديد فور تحديده. نعتذر عن أي إزعاج قد يسببه هذا التغيير، ونشكر لكم تفهمكم وتعاونكم المستمر.  

مع خالص الشكر والتقدير،  
جمعية المحتوى الإسلامي


In [11]:
from langgraph.types import Command

HUMAN_TEXT = (
    "النص المعتمد من المشرف البشري: فعالية السبت مؤجلة أسبوعًا، "
    "وسيُعلن الموعد الجديد عبر قنوات الجمعية."
)

done = app.invoke(Command(resume=HUMAN_TEXT), THREAD_MAIL)

print("الرد النهائي:", done["reply"])
print("مطابق لنص البشري حرفيًا:", done["reply"] == HUMAN_TEXT)
print("مسار صندوق الصادر:", done["outbox_path"])
print("---- محتوى الملف المكتوب ----")
print((PROJECT_ROOT / done["outbox_path"]).read_text(encoding="utf-8"))


Deserializing unregistered type munassiq.app.DraftVerdict from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('munassiq.app', 'DraftVerdict')]


Deserializing unregistered type munassiq.tools.TriageDecision from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('munassiq.tools', 'TriageDecision')]


الرد النهائي: النص المعتمد من المشرف البشري: فعالية السبت مؤجلة أسبوعًا، وسيُعلن الموعد الجديد عبر قنوات الجمعية.
مطابق لنص البشري حرفيًا: True
مسار صندوق الصادر: data/outbox/sent-20260819-231027-588407.txt
---- محتوى الملف المكتوب ----
النص المعتمد من المشرف البشري: فعالية السبت مؤجلة أسبوعًا، وسيُعلن الموعد الجديد عبر قنوات الجمعية.


## القسم 6 — Functional API واستراتيجيتا الخطأ

`@entrypoint` يعرّف الرحلة، و`@task` يعرّف وحدةً تُنفَّذ مرة وتُحفظ نتيجتها في
الـcheckpointer. وجسم الـentrypoint **غراءٌ نقي**: شرطٌ، ونداءات `@task`، وجمع
نتائجها — لا أكثر. والسبب ميكانيكي لا تجميلي: عند الاستئناف بعد `interrupt`
يُعاد تنفيذ جسم الـentrypoint من أوله، بينما نتائج الـ`@task` المكتملة تُقرأ
من الـcheckpointer ولا تُعاد. فسطرٌ ينادي نموذجًا خارج `@task` يعني فاتورةً
مكررة وناتجًا مختلفًا عمّا بُني عليه القرار.

والاستراتيجيتان مختلفتان لأن **مالك الإصلاح** مختلف لا لأن شدة الخطأ مختلفة:

1. **العابر Transient** — لا أحد يملك إصلاحه والوقت وحده يصلحه، فالعلاج
   `RetryPolicy` على المهمة نفسها لا حلقة `try/except` داخل جسمها: LangGraph
   هو من يدير المحاولات والمهل فتظهر مرقّمةً في الـcheckpointer وفي الأثر،
   بينما الحلقة اليدوية تُخفي الفشل داخل نداءٍ واحد ناجح ظاهريًا. وبعد
   استنفاد المحاولات **ينتشر** الاستثناء ولا يُبتلَع.
2. **خطأ المدخل LLM-recoverable** — النموذج نفسه أخطأ، والتكرار الأعمى يعيد
   المدخل نفسه فيقع الخطأ نفسه إلى الأبد. فالعلاج أن يدخل نصُّ الخطأ **سياقَ
   النموذج** رسالةً تصحيحية، فيصير الخطأ معلومةً يتعلّم منها لا حائطًا يرتطم به.

وخلايا عرض الأخطاء تطبع `type(e).__name__` والرسالة فقط — **لا traceback
خام**: هو يحمل مسارات ملفاتٍ مطلقة تكشف اسم المستخدم وبنية الجهاز، وتبقى
محفوظةً في مخرجات النوتبوك.


In [12]:
from munassiq.workers import fetch_external_resource, run_reliability_task

attempts = {"count": 0}


def flaky_fetch(resource: str) -> str:
    """يفشل مرتين بانقطاع مفتعل ثم ينجح — دالة محقونة، بلا أي نداء نموذج."""
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ConnectionError("انقطاع مفتعل في المورد الخارجي")
    return f"محتوى {resource}"


value = run_reliability_task(
    fetch_external_resource, "قائمة المتطوعين", fetcher=flaky_fetch
)

print("عدد المحاولات التي جرت فعلًا:", attempts["count"])
print("الناتج بعد نجاح المحاولة الثالثة:", value)


عدد المحاولات التي جرت فعلًا: 3
الناتج بعد نجاح المحاولة الثالثة: محتوى قائمة المتطوعين


In [13]:
from munassiq.workers import run_tool_with_llm_recovery

ALLOWED_SLOTS = ("SAT-2026-08-22", "SUN-2026-08-23")


def strict_slot_tool(slot: str) -> str:
    """أداة لا تقبل إلا رمزًا من قائمة مغلقة — ورسالة خطئها هي ما يُعلّم النموذج.

    القائمة **غائبة** عن تعليمات النموذج عمدًا: السبيل الوحيد إلى المدخل
    الصحيح هو نص الخطأ العائد من الأداة.
    """
    if slot not in ALLOWED_SLOTS:
        raise ValueError(
            f"الرمز «{slot}» غير موجود؛ الرموز المتاحة: {', '.join(ALLOWED_SLOTS)}"
        )
    return f"حُجز الموعد {slot}."


try:
    outcome = run_reliability_task(
        run_tool_with_llm_recovery,
        "احجز موعد فعالية السبت. أخرج رمز الموعد وحده بلا أي شرح.",
        tool=strict_slot_tool,
    )
    print("عدد نداءات الأداة:", outcome["attempts"])
    print("الأخطاء المصحَّحة (النوع والرسالة فقط):")
    for line in outcome["errors"]:
        print("   ", line)
    print("الناتج بعد التصحيح:", outcome["result"])
except Exception as error:  # النوع فقط — traceback يحمل مسارات مطلقة
    print("لم يُصحَّح المدخل في هذه التشغيلة:", type(error).__name__)


عدد نداءات الأداة: 2
الأخطاء المصحَّحة (النوع والرسالة فقط):
    ValueError: الرمز «الرجاء تزويدي برمز الموعد المطلوب.» غير موجود؛ الرموز المتاحة: SAT-2026-08-22, SUN-2026-08-23
الناتج بعد التصحيح: حُجز الموعد SAT-2026-08-22.


## القسم 7 — النمط المسمّى: Evaluator-Optimizer

النمط المطبَّق في مسار المراسلات اسمه **Evaluator-Optimizer**: مولِّدٌ يكتب
المسودة، ثم مقيِّمٌ يحكم عليها حكمًا **مهيكلًا** (`DraftVerdict` بحقول `score`
و`approved` و`feedback`)، ثم محسِّنٌ يعيد الكتابة بالملاحظات إن رُفضت، ثم
يُعاد التقييم — بسقفٍ صلب جولتين، فمقيّمٌ لا يقتنع أبدًا لا يُدير حلقةً بلا
نهاية.

**ولماذا يناسب المراسلات بالذات**: الرسالة الصادرة باسم الجمعية لها معيار
قبولٍ يمكن النطق به — تنقل ما طُلب كاملًا، بلا زيادةٍ لم تَرِد في الطلب،
بعربية فصيحة موجزة، بتحيةٍ وخاتمة. وهذا بالضبط شرط نجاح Evaluator-Optimizer:
ناقدٌ يقدر أن يقول ما يُصلَح وكيف، لا مجرد «حسّنها». ولأن آخر الخط مراجعٌ
بشري، فالحلقة كلها تسبق الوقوف: يُعرض على البشري نصٌّ نُقّح لا مسوّدةٌ خام.

والقرار يُقرأ من الحقل المهيكل `approved` وحده لا بالبحث عن كلمةٍ في نص
الملاحظات — فحكمُ رفضٍ قد يذكر «معتمدة» نفيًا أو اقتباسًا، فيمرّ ما كان يجب
أن يُحسَّن. والحلقة مطويّة داخل `@task` واحدة، فتُستعاد وحدةً واحدة من
الـcheckpointer عند الاستئناف بدل أن تُعاد جولاتها.


In [14]:
THREAD_EVAL = {"configurable": {"thread_id": "nb-eval-opt"}}

paused_eval = app.invoke(
    {
        "request": "اكتب رسالة شكر لفريق النشر على إنجاز جدول الأسبوع",
        "user_id": USER_ID,
    },
    THREAD_EVAL,
)

loop_payload = paused_eval["__interrupt__"][0].value
print("عدد جولات التقييم:", loop_payload["evaluation_rounds"])
print("درجة المقيّم للمسودة المعروضة:", loop_payload["evaluation_score"])
print("---- المسودة بعد الحلقة ----")
print(loop_payload["draft"])

# الاعتماد كما هي — والحصيلة نفسها تعود في ناتج الرحلة المكتملة.
approved_run = app.invoke(Command(resume=loop_payload["draft"]), THREAD_EVAL)
print("حصيلة الحلقة في الناتج النهائي:", approved_run["evaluation"])


عدد جولات التقييم: 1
درجة المقيّم للمسودة المعروضة: 9
---- المسودة بعد الحلقة ----
تحية طيبة،  

نتقدم بجزيل الشكر لفريق النشر على إنجاز جدول الأسبوع بدقة وإتقان، مما يعكس حرصكم المستمر على تقديم محتوى متميز ومهني.  

مع خالص التقدير،  
جمعية المحتوى الإسلامي.
حصيلة الحلقة في الناتج النهائي: {'rounds': 1, 'score': 9}


## القسم 8 — تتبع LangSmith

`since` يُلتقط **قبل** نداء النموذج، فالـrun الذي يعود لا يمكن أن يكون بقيّةَ
تشغيلةٍ سابقة — وهذا ما يجعل الانتظار دليلًا على أن التتبع يعمل الآن، لا على
أنه عمل يومًا ما. والانتظار polling بمهلة لا نومٌ ثابت: الـrun قد يظهر بعد
ثانية وقد يتأخر عشرًا، فالنوم الثابت إمّا يُبطئ كل تشغيلة أو يسقط عشوائيًا.

ولا يُطبع من الـrun إلا معرّفه واسمه وحالته: لا مدخلات ولا مخرجات (فيها نصوص
الأعضاء والمراسلات) ولا روابط موقّعة.

> **الفخ**: LangChain تقرأ `LANGCHAIN_TRACING_V2` حرفيًا. والاسم
> `LANGSMITH_TRACING_V2` يبدو صحيحًا تمامًا ولا يشتكي منه أحد — لا LangChain
> ولا LangSmith ولا مفسّر بايثون — لكن التتبع حينها **مطفأ**، ولا يصل أي
> trace، ولا يظهر شيء على اللوحة. الفشل صامت: لا استثناء ولا تحذير، فقط
> مشروعٌ فارغ يُكتشف متأخرًا.


In [15]:
import datetime as dt

from munassiq.config import get_llm
from munassiq.tracing import wait_for_recent_run

since = dt.datetime.now(dt.timezone.utc)  # قبل النداء لا بعده
get_llm().invoke("تحقق من وصول التتبع")

run = wait_for_recent_run(since=since, timeout_s=60, poll_s=5)
print("معرّف الـrun:", run["id"])
print("اسم الـrun:", run["name"])
print("حالة الـrun:", run["status"])


معرّف الـrun: 01a01ba6-af30-74c2-b77f-5eed5a050ac9
اسم الـrun: ChatOpenAI
حالة الـrun: pending


### ملاحظة من الـtrace

**ما أظهره الـtrace فعلًا** (من runs مشروع munassiq-capstone، تشغيلة 19 أغسطس): حلقة Evaluator-Optimizer هي عنق الزجاجة الحقيقي — صياغة المسودة (compose_draft) استغرقت 3.3 ثوانٍ بينما تقييمها (evaluate_draft) بلغ 21.7 ثانية، أي أن الحَكم المهيكل أبطأ من الكاتب بنحو سبع مرات لأن إخراج Pydantic المقيد يجبر النموذج على تخطيط أدق. وأظهر الـtrace أيضًا حلقة تصحيح الأداة (run_tool_with_llm_recovery) بـ35.3 ثانية تشمل نداءي نموذج متتاليين — المحاولة الفاشلة ورسالة التصحيح — وهو بالضبط ما صممناها له.


## الخاتمة — خريطة الروبرك إلى خلايا هذا النوتبوك

| قسم الروبرك | أين دليله هنا |
|---|---|
| 1 — أساسيات الوكيل | القسم 1: أدواتٌ تغيّر `CALENDAR` بمعاملاتها، و`TriageDecision` مخرجًا مهيكلًا |
| 2 — المشرف والتوجيه | القسم 2: نداءات `transfer_to_*` مطبوعةً من رسائل المشرف |
| 3 — RAG | القسم 3: تبرير Agentic مقابل 2-Step وHybrid، والمقطع المسترجَع بمصدره، ثم جواب عامل المعرفة |
| 4 — الذاكرة | القسم 4: كتابةٌ في `nb-thread-1`، ثم `store.search`، ثم استدعاءٌ من `nb-thread-2`، ثم قِصر المدى على الـthread نفسه |
| 5 — الوقوف البشري | القسم 5: خلية `__interrupt__` بحمولتها، ثم خلية `Command(resume=...)` وملف صندوق الصادر |
| 6 — Functional API والموثوقية | القسم 6: `RetryPolicy` بعدّاد محاولات حقيقي، وتصحيح مدخل الأداة بنص الخطأ |
| 7 — النمط المسمّى | القسم 7: Evaluator-Optimizer بعدد الجولات ودرجة المقيّم |
| 8 — التتبع | القسم 8: `id` و`name` و`status` لـrun وُلد بعد `since` |

**الاختبارات**: المنطق كله في `src/munassiq/` ومغطّى بـ`pytest` تحت `tests/`
(‏`pytest -m "not api"` يشغّل ما لا يستهلك حصة نموذج). ومع هذا التنفيذ يُرفع
وسم `xfail` عن `tests/test_integration.py::test_capstone_end_to_end` فيصير
أخضر بلا وسم — وهو الصياغة التنفيذية لمعيار نجاح المشروع.

**بوابة التسرب**: `python tools/leak_scan.py` يفحص هذا النوتبوك وكل ملف
متتبَّع في git عن مفاتيح ومسارات جهاز وأسماء، ويخرج بـ1 عند أي تطابق — يُشغَّل
قبل أي دفع.
